In [1]:
"""
R2R Phase 1 — Step 2d: Build Final Task Register
==================================================
Merges the consensus-resolved tasks with the auto-resolved tasks to produce
the complete validated R2R Task Register — the output of Phase 1.

Inputs:
  outputs/consensus_resolved.csv   — from 02c_tally_and_vote.py
                                     (all needs_human_review must be False
                                     or manually filled before running this)

  auto_resolved.csv                — tasks that reached 4/4 unanimous
                                     agreement in the initial round and
                                     did not enter the consensus pipeline

Outputs (written to outputs/):
  final_task_register.csv          — complete validated register, one row
                                     per task, with provenance columns

  final_task_register_summary.csv  — task counts per R2R subprocess +
                                     out-of-scope, for Phase 2 staging

Usage:
    python 02d_build_final_register.py

Before running:
  1. Open outputs/consensus_resolved.csv
  2. Filter needs_human_review = True
  3. Manually set final_label for any remaining 2/2 splits
  4. Save and re-run this script
"""

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

CONSENSUS_RESOLVED_XLSX = "june23_consensus_resolved.xlsx"
AUTO_RESOLVED_CSV      = "auto_resolved.csv"

OUTPUT_DIR = Path("final register")
OUTPUT_DIR.mkdir(exist_ok=True)

# Column name in auto_resolved.csv that holds the final label.
# Check your file — common names: "final_label", "classification"
AUTO_RESOLVED_LABEL_COL = "final_label"

# Column in auto_resolved.csv that holds the task statement
AUTO_RESOLVED_STMT_COL  = "task_statement"

# Column in auto_resolved.csv that holds the task id
AUTO_RESOLVED_ID_COL    = "task_id"

# ---------------------------------------------------------------------------
# VALID LABELS (for validation check)
# ---------------------------------------------------------------------------

VALID_LABELS = {
    "Master Data Management",
    "Record",
    "Close & Consolidation",
    "Report",
    "Analysis & Business Insight",
    "Planning & Budgeting",
    "OUT_OF_SCOPE",
}

# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

def load_consensus_resolved(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath, dtype=str).fillna("")
    required = {"task_id", "task_statement", "final_label", "needs_human_review"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"consensus_resolved.csv missing columns: {missing}")
    return df


def load_auto_resolved(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, dtype=str).fillna("")
    required = {AUTO_RESOLVED_ID_COL, AUTO_RESOLVED_STMT_COL, AUTO_RESOLVED_LABEL_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"auto_resolved.csv missing expected columns: {missing}\n"
            f"Available: {list(df.columns)}\n"
            f"Update AUTO_RESOLVED_LABEL_COL / AUTO_RESOLVED_STMT_COL / "
            f"AUTO_RESOLVED_ID_COL at the top of this script."
        )
    return df


def check_no_unresolved(consensus_df: pd.DataFrame) -> None:
    """Abort if any tasks still have needs_human_review = True."""
    mask = consensus_df["needs_human_review"].str.strip().str.lower() == "true"
    unresolved = consensus_df.loc[mask, "task_id"].tolist()
    if unresolved:
        print(
            f"\n  ERROR: {len(unresolved)} task(s) in consensus_resolved.csv still have "
            f"needs_human_review = True and no final_label.\n"
            f"  Please fill in final_label manually before running this script.\n"
            f"  Affected task IDs: {unresolved}"
        )
        sys.exit(1)


def build_consensus_block(df: pd.DataFrame) -> pd.DataFrame:
    """Select and rename columns needed for the final register."""
    cols = {
        "task_id":           "task_id",
        "task_statement":    "task_statement",
        "final_label":       "final_label",
        "resolution_method": "resolution_method",
        "vote_tally":        "vote_tally_consensus",
        "human_consensus":   "human_consensus",
        "claude_consensus":  "claude_consensus",
        "openai_consensus":  "openai_consensus",
        "gemini_consensus":  "gemini_consensus",
    }
    available = {k: v for k, v in cols.items() if k in df.columns}
    out = df[list(available.keys())].rename(columns=available).copy()

    # Columns that come from auto-resolved won't exist here — fill with blanks
    for col in ["vote_tally_initial"]:
        if col not in out.columns:
            out[col] = ""

    out["pipeline_stage"] = "consensus_round"
    return out


def build_auto_block(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise auto-resolved CSV to the register schema."""
    out = pd.DataFrame()
    out["task_id"]       = df[AUTO_RESOLVED_ID_COL].str.strip()
    out["task_statement"] = df[AUTO_RESOLVED_STMT_COL].str.strip()
    out["final_label"]   = df[AUTO_RESOLVED_LABEL_COL].str.strip()

    # Carry over vote_tally_initial if present
    if "vote_tally_initial" in df.columns:
        out["vote_tally_initial"] = df["vote_tally_initial"]
    elif "vote_tally" in df.columns:
        out["vote_tally_initial"] = df["vote_tally"]
    else:
        out["vote_tally_initial"] = ""

    # resolution_method — use existing column or infer from unanimous 4/4 label
    if "resolution_method" in df.columns:
        out["resolution_method"] = df["resolution_method"]
    else:
        out["resolution_method"] = "initial_unanimous_4_4"

    # Consensus-round columns are blank for auto-resolved tasks
    for col in [
        "vote_tally_consensus",
        "human_consensus", "claude_consensus",
        "openai_consensus", "gemini_consensus",
    ]:
        out[col] = ""

    out["pipeline_stage"] = "auto_resolved"
    return out


def validate_labels(df: pd.DataFrame, source: str) -> None:
    """Warn about any final_label values not in the allowed set."""
    bad = df.loc[~df["final_label"].isin(VALID_LABELS), ["task_id", "final_label"]]
    if not bad.empty:
        print(
            f"\n  WARNING ({source}): {len(bad)} row(s) have unexpected final_label values:"
        )
        for _, row in bad.iterrows():
            print(f"    task_id={row['task_id']}  label='{row['final_label']}'")


def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Count tasks per subprocess, split by pipeline stage."""
    summary = (
        df.groupby(["final_label", "pipeline_stage"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    # Add a total column
    stage_cols = [c for c in summary.columns if c != "final_label"]
    summary["total"] = summary[stage_cols].sum(axis=1)
    summary = summary.sort_values("total", ascending=False).reset_index(drop=True)
    return summary

# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main() -> None:
    # --- Load inputs ---
    print(f"Loading consensus-resolved tasks from: {CONSENSUS_RESOLVED_CSV}")
    consensus_df = load_consensus_resolved(CONSENSUS_RESOLVED_CSV)
    print(f"  {len(consensus_df)} tasks")

    check_no_unresolved(consensus_df)

    print(f"Loading auto-resolved tasks from: {AUTO_RESOLVED_CSV}")
    auto_df = load_auto_resolved(AUTO_RESOLVED_CSV)
    print(f"  {len(auto_df)} tasks")

    # --- Normalise to register schema ---
    consensus_block = build_consensus_block(consensus_df)
    auto_block      = build_auto_block(auto_df)

    # --- Check for task_id overlaps (should never happen) ---
    overlap = set(consensus_block["task_id"]) & set(auto_block["task_id"])
    if overlap:
        print(
            f"\n  WARNING: {len(overlap)} task_id(s) appear in both files — "
            f"keeping the consensus-round version.\n"
            f"  IDs: {sorted(overlap)[:10]}"
        )
        auto_block = auto_block[~auto_block["task_id"].isin(overlap)]

    # --- Concatenate ---
    register = pd.concat([auto_block, consensus_block], ignore_index=True)
    register = register.sort_values("task_id").reset_index(drop=True)

    # --- Validate labels ---
    validate_labels(consensus_block, "consensus_resolved")
    validate_labels(auto_block, "auto_resolved")

    # --- Write final register ---
    register_path = OUTPUT_DIR / "final_task_register.csv"
    register_cols = [
        "task_id", "task_statement", "final_label",
        "pipeline_stage", "resolution_method",
        "vote_tally_initial", "vote_tally_consensus",
        "human_consensus", "claude_consensus",
        "openai_consensus", "gemini_consensus",
    ]
    # Only output columns that exist
    out_cols = [c for c in register_cols if c in register.columns]
    register[out_cols].to_csv(register_path, index=False)
    print(f"\n  Saved: {register_path}  ({len(register)} tasks total)")

    # --- Write summary ---
    summary = build_summary(register)
    summary_path = OUTPUT_DIR / "final_task_register_summary.csv"
    summary.to_csv(summary_path, index=False)
    print(f"  Saved: {summary_path}")

    # --- Print summary to console ---
    print(f"\n=== Final Task Register — Distribution ===")
    print(f"{'Subprocess':<35} {'auto':>6} {'consensus':>10} {'total':>7}")
    print("-" * 62)
    for _, row in summary.iterrows():
        auto_n = row.get("auto_resolved", 0)
        cons_n = row.get("consensus_round", 0)
        print(f"  {row['final_label']:<33} {auto_n:>6} {cons_n:>10} {row['total']:>7}")
    print("-" * 62)
    total_in_scope = register[register["final_label"] != "OUT_OF_SCOPE"].shape[0]
    total_all      = len(register)
    print(f"  {'TOTAL (all)':<33} {total_all:>24}")
    print(f"  {'TOTAL (in-scope R2R)':<33} {total_in_scope:>24}")

    print(f"\nPhase 1 complete. Proceed to Phase 2 with {register_path.name}.")


if __name__ == "__main__":
    main()

Loading consensus-resolved tasks from: june23_consensus_resolved.xlsx
  81 tasks
Loading auto-resolved tasks from: auto_resolved.csv
  131 tasks

  Saved: final register/final_task_register.csv  (212 tasks total)
  Saved: final register/final_task_register_summary.csv

=== Final Task Register — Distribution ===
Subprocess                            auto  consensus   total
--------------------------------------------------------------
  OUT_OF_SCOPE                          92         29     121
  Record                                14         17      31
  Planning & Budgeting                   6         16      22
  Report                                 6          7      13
  Analysis & Business Insight            6          6      12
  Master Data Management                 5          3       8
  Close & Consolidation                  2          3       5
--------------------------------------------------------------
  TOTAL (all)                                            212
  TO